## Sequence-to-sequence model for German-English Translation

In [ ]:
import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from nltk.translate.bleu_score import sentence_bleu
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.nn.functional as F
from typing import Iterable
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import math
from tqdm.notebook import tqdm, trange
import random
from sklearn.model_selection import train_test_split
import multiprocessing
import re
import string
import unicodedata
import csv

### Download and prepare the Tatoeba dataset

In [ ]:
dataset_file = "deu-eng/deu.txt"

In [ ]:
def process_tab_delimited_file(filename):
    data = []
    with open(filename, newline='', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter='\t')
        for row in reader:
            if len(row) >= 2:  # Ensure there are at least two elements (English and Other Language)
                other_language, english = row[1], row[0]
                data.append([other_language, english])
    return data

In [ ]:
train_pairs = process_tab_delimited_file(dataset_file)

In [ ]:
print(len(train_pairs))

In [ ]:
train_pairs

In [ ]:
def include_pair(pair, max_length):
    return len(pair[0].split(' ')) <= max_length and \
        len(pair[1].split(' ')) <= max_length

def filter_pairs(pairs, max_length):
    return [pair for pair in tqdm(pairs) if include_pair(pair, max_length)]

In [ ]:
max_len = 10

In [ ]:
filtered_pairs = filter_pairs(train_pairs, max_len)

In [ ]:
len(filtered_pairs)

In [ ]:
def unicode_to_ascii(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')

def normalize(s):
    s = unicode_to_ascii(s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = s.lower().strip()
    s = re.sub(r'\d+', '', s)
    s = re.sub(r'[^A-Za-z0-9 ]+', '', s)
    return s.strip()

def process_pairs(pairs):
    return [[normalize(pair[0]), normalize(pair[1])] for pair in tqdm(pairs)]

In [ ]:
processed_pairs = process_pairs(filtered_pairs)

In [ ]:
processed_pairs[:10]

In [ ]:
print(len(processed_pairs))

In [ ]:
ftrain_pairs, ftest_pairs = train_test_split(processed_pairs, test_size=0.05)

In [ ]:
def yield_tokens(data, idx, token_lang):
    tokenizer =  get_tokenizer ('spacy', language=token_lang)
    for item in tqdm(data):
        text = item[idx]
        tokenized_text = tokenizer(text)
        yield tokenized_text  

In [ ]:
german_vocab = build_vocab_from_iterator(
    yield_tokens(ftrain_pairs, 0, 'de_core_news_sm'),
    specials=['<unk>', '<pad>', '<bos>', '<eos>'],
    min_freq=20)
german_vocab.set_default_index(german_vocab['<unk>'])

In [ ]:
# Tokenize and build vocab for English
english_vocab = build_vocab_from_iterator(
    yield_tokens(ftrain_pairs, 1, 'en_core_web_sm'),
    specials=['<unk>', '<pad>', '<bos>', '<eos>'],
    min_freq=20)
english_vocab.set_default_index(english_vocab['<unk>'])

In [ ]:
print("German vocabulary size:", len(german_vocab))
print("English vocabulary size:", len(english_vocab))

In [ ]:
def tokenize_and_transform(data_iter, source_language, target_language, source_vocab, target_vocab):
    source_tokenizer = get_tokenizer('spacy', language=source_language)
    target_tokenizer = get_tokenizer('spacy', language=target_language)
    tokenized_src = []
    tokenized_trg = []

    for item in tqdm(data_iter):
        source_text = item[0]
        target_text = item[1] 
        source_tokenized = source_tokenizer(source_text)
        target_tokenized = target_tokenizer(target_text)
        source_indexed = [source_vocab[token] for token in source_tokenized]
        target_indexed = [target_vocab[token] for token in target_tokenized]
        tokenized_src.append(source_indexed)
        tokenized_trg.append(target_indexed)

    return tokenized_src, tokenized_trg

In [ ]:
tgerman, tenglish = tokenize_and_transform(ftrain_pairs, 'de_core_news_sm', 'en_core_web_sm', 
                                           german_vocab, english_vocab)

In [ ]:
print(len(tgerman))
print(len(tenglish))

In [ ]:
ttgerman, ttenglish = tokenize_and_transform(ftest_pairs, 'de_core_news_sm', 'en_core_web_sm', 
                                            german_vocab, english_vocab)

In [ ]:
class TokenizedPairsDataset(Dataset):
    def __init__(self, source_sequences, target_sequences, src_vocab, trg_vocab):
        assert len(source_sequences) == len(target_sequences), "Source and target sequences must have the same length."
        self.src_seq = [torch.tensor([src_vocab['<bos>']] + seq + [src_vocab['<eos>']]) for seq in source_sequences]
        self.trg_seq = [torch.tensor([trg_vocab['<bos>']] + seq + [trg_vocab['<eos>']]) for seq in target_sequences]
        self.src_pad_idx = src_vocab['<pad>']
        self.trg_pad_idx = trg_vocab['<pad>']                            

    def __len__(self):
        return len(self.src_seq)

    def __getitem__(self, idx):
        return self.src_seq[idx], self.trg_seq[idx]

    def collate_fn(self, batch):
        src_batch, trg_batch = zip(*batch)
        src_padded = pad_sequence(src_batch, padding_value=self.src_pad_idx, batch_first=True)
        trg_padded = pad_sequence(trg_batch, padding_value=self.trg_pad_idx, batch_first=True)
        return src_padded, trg_padded

In [ ]:
train_dataset = TokenizedPairsDataset(tgerman, tenglish, german_vocab, english_vocab)
test_dataset = TokenizedPairsDataset(ttgerman, ttenglish, german_vocab, english_vocab)

In [ ]:
batch_size = 64

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, 
                              shuffle=True, collate_fn=train_dataset.collate_fn, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, 
                             shuffle=True, collate_fn=test_dataset.collate_fn, drop_last=True)

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Build the Encoder-Decoder model

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_size, num_layers=1, dropout=0.0):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_dim, emb_dim) 
        self.lstm = nn.LSTM(emb_dim, hidden_size, num_layers, dropout=dropout, batch_first=True)

    def forward(self, src):
        # src is of shape [batch_size, sequence_len]
        embedded = self.embedding(src)  # [batch_size, sequence_len, emb_dim]
        outputs, hidden = self.lstm(embedded)
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_size, output_size, num_layers=1, dropout=0.0):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(output_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        #input = input.unsqueeze(1)  # [batch_size] -> [batch_size, 1]
        input = self.embedding(input)  # [batch_size, 1, emb_dim]
        output, hidden = self.lstm(input, hidden)
        output = self.out(output)
        return output, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device, bos_token):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.bos_token = bos_token

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.out.out_features
        #outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(device)
        encoder_output, encoder_hidden = self.encoder(src)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(self.bos_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        
        for i in range(trg_len):
            decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)
            teacher_force = random.random() < teacher_forcing_ratio
            
            if teacher_force:
                # Teacher forcing: Feed the target as the next input
                decoder_input = trg[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs

In [ ]:
hidden_size = 128
num_layers = 2
embedding_dim = 128
input_size = len(german_vocab)
output_size = len(english_vocab)
dropout = 0.5
epochs = 10
lr = 0.001

In [ ]:
enc = Encoder(input_size, embedding_dim, hidden_size, num_layers=num_layers, dropout=dropout)

In [ ]:
dec = Decoder(hidden_size, embedding_dim, hidden_size, output_size, num_layers=num_layers, dropout=dropout)

In [ ]:
dec

In [ ]:
seq2seq = Seq2Seq(enc, dec, device, english_vocab['<bos>']).to(device)

In [ ]:
loss_fn = nn.NLLLoss()
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

In [ ]:
def train_seq2seq_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, device, clip=0.25):
    train_losses = []
    test_losses = []
    initial_teacher_forcing_ratio = 0.5
    half_life_steps = 10

    for epoch in range(epochs):
        current_teacher_forcing_ratio = initial_teacher_forcing_ratio * (0.5 ** (epoch / half_life_steps))
        model.train()
        train_loss = 0.0
        total_train = 0

        # Training loop
        print("Training")
        for src, trg in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            src = src.to(device)
            trg = trg.to(device)

            # Zero the gradients
            model.zero_grad()

            # Forward pass
            output = model(src, trg)

            # Calculate the loss
            loss = loss_fn(
                output.view(-1, output.size(-1)),
                trg.view(-1)
            )
            train_loss += loss.item() * src.size(0)
            total_train += trg.size(0)

            # Backward pass and optimize
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

        train_loss /= total_train
        train_losses.append(train_loss)

        # Evaluation loop
        model.eval()
        test_loss = 0.0
        total_test = 0
        print("Evaluating")
        with torch.no_grad():
            for src, trg in test_loader:
                src = src.to(device)
                trg = trg.to(device)
                
                # Forward pass
                output = model(src, trg, 0)  # turn off teacher forcing
                # Calculate the loss
                loss = loss_fn(
                    output.view(-1, output.size(-1)),
                    trg.view(-1)
                )
                test_loss += loss.item() * src.size(0)
                total_test += trg.size(0)

        test_loss /= total_test
        test_losses.append(test_loss)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

    return train_losses, test_losses

In [ ]:
train_losses, test_losses = train_seq2seq_model(seq2seq, train_dataloader, test_dataloader, 
                                                loss_fn, optimizer, epochs, device)

In [ ]:
model_path = "seq2seq_de.pth"  # Replace with your desired file path

In [ ]:

torch.save(seq2seq.state_dict(), model_path)

### Translation and BLEU score

In [ ]:
def translate(model, src_sentence, src_vocab, trg_vocab, src_tokenizer, device, max_length=10):
    model.eval()
    s = normalize(src_sentence)
    tokens = [src_vocab['<bos>']] + [src_vocab[token] for token in src_tokenizer(s)] + [src_vocab['<eos>']]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)  # Shape: [1, len(tokens)]
    
    with torch.no_grad():
        _, encoder_hidden = model.encoder(src_tensor)
        decoder_input = torch.empty(1, 1, dtype=torch.long, device=device).fill_(trg_vocab['<bos>'])
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        
        
        for i in range(max_length):
            decoder_output, decoder_hidden = model.decoder(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)
            _, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze(-1).detach()
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_ids = torch.argmax(decoder_outputs, dim=-1).squeeze()
        decoded_words = []
        for idx in decoder_ids:
            if idx.item() == trg_vocab['<eos>']:
                decoded_words.append('<eos>')
                break
            decoded_words.append(trg_vocab.get_itos()[idx.item()])
    return decoded_words

In [ ]:
src_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')

In [ ]:
src_sentence = "Ich mag Eis"

In [ ]:
translation = translate(seq2seq, src_sentence, german_vocab, 
                        english_vocab, src_tokenizer, device)

In [ ]:
translation

In [ ]:
bleu_score = 0
for src, trg in ftest_pairs: 
    translation = translate(seq2seq, src, german_vocab, english_vocab, src_tokenizer, device)
    reference = trg.split()
    bleu_score += sentence_bleu([reference], translation)

average_bleu_score = bleu_score / len(test_dataset)
print(f"Average BLEU Score: {average_bleu_score}")